<style>
/* 수업용 노트북: 기본값보다 조금 작고 촘촘하게 표시합니다. */
.jp-MarkdownOutput, .markdown-body {
    font-size: 0.94em !important;
    line-height: 1.68 !important;
}
.jp-MarkdownOutput h1, .markdown-body h1 { font-size: 1.72em !important; }
.jp-MarkdownOutput h2, .markdown-body h2 { font-size: 1.38em !important; }
.jp-MarkdownOutput h3, .markdown-body h3 { font-size: 1.14em !important; }
.jp-CodeCell, .jp-OutputArea-output, .cell.code_cell, .output_area {
    font-size: 0.92em !important;
}
.jp-MarkdownOutput table, .markdown-body table { font-size: 0.92em !important; }
.jp-MarkdownOutput blockquote, .markdown-body blockquote {
    border-left: 4px solid #4c78a8;
    padding-left: 0.9em;
    color: #4b5563;
}
</style>

# 처음 시작하는 CIFAR-10 백본 비교 실습

이 노트북은 이미지 분류를 처음 접하는 사람도 `모델 선택 → 데이터 확인 → 학습 → 검증 → 최종 테스트` 과정을 순서대로 실행할 수 있도록 만든 통합 실습입니다.

처음이라면 다음 순서를 권장합니다.

1. `foundations.ipynb`: 퍼셉트론과 MLP의 원리
2. `cnn_internals.ipynb`: AlexNet 내부 특징맵과 수용영역
3. `main.ipynb`: 여러 CNN·Transformer 백본의 실제 비교

### 실행 모드

| 모드 | 설정 | 목적 |
|---|---|---|
| 빠른 체험 | `QUICK_RUN = True` | 설치와 전체 파이프라인 확인 |
| 본 학습 | `QUICK_RUN = False` | 전체 CIFAR-10으로 의미 있는 비교 |

> 빠른 체험의 정확도는 최종 모델 성능이 아닙니다. 처음에는 모든 셀이 오류 없이 실행되고 결과 파일이 저장되는지 확인하는 데 집중하세요.

## 0. 필요한 기능 불러오기

백본 파일을 하나씩 직접 import하지 않고 Registry를 사용합니다. Registry는 모델 ID와 실제 생성 함수를 연결해 주므로, 아래 설정에서 `MODEL_ID`만 바꾸어 같은 학습 파이프라인을 재사용할 수 있습니다.

이 셀에서 불러오는 기능은 네 범주입니다.

- **설정**: 데이터와 학습 조건을 하나의 객체로 기록합니다.
- **모델**: 문자열 ID로 백본을 생성합니다.
- **학습·평가**: 모든 모델에 같은 반복문을 적용합니다.
- **시각화**: 데이터 분할, 학습 곡선, confusion matrix를 표시합니다.

In [ ]:
# 모델 구조와 파라미터 수를 보기 위한 도구입니다.
from torchinfo import summary

# cifar10_lab의 공통 설정, 모델 Registry, 학습/평가 기능을 불러옵니다.
from cifar10_lab import (
    DataConfig,
    ExperimentConfig,
    TrainConfig,
    create_model,
    detect_environment,
    evaluate_model_detailed,
    experiment_id,
    format_model_catalog,
    get_lab_paths,
    load_cifar10_data,
    load_model_weights,
    resolve_data_dir,
    set_global_seed,
    train_model,
)

# 그래프를 그리는 기능은 시각화 모듈에서 따로 불러옵니다.
from cifar10_lab.visualization import (
    plot_test_results,
    plot_training_history,
    visualize_data_overview,
)

print("라이브러리 불러오기 완료")

## 1. 실험 설정하기 — 처음에는 두 값만 변경하세요

- `MODEL_ID`: 실행할 백본 이름입니다. 첫 실행은 `resnet18`을 권장합니다.
- `QUICK_RUN`: `True`는 일부 데이터와 1 epoch, `False`는 전체 데이터와 더 긴 학습을 사용합니다.

나머지 설정의 의미는 다음과 같습니다.

| 설정 | 의미 | 비교 실험에서의 역할 |
|---|---|---|
| `batch_size` | 한 번에 처리할 이미지 수 | 메모리 사용량과 갱신 횟수에 영향 |
| `seed` | 난수 시작점 | 데이터 분할과 초기화 재현 |
| `val_ratio` | 학습 데이터 중 Validation 비율 | 모델 선택용 데이터 크기 결정 |
| `learning_rate` | 한 번의 가중치 이동 크기 | 너무 크면 불안정, 너무 작으면 느림 |
| `epochs` | 전체 Train 데이터를 반복하는 횟수 | 학습 시간과 수렴 정도 결정 |

공정한 백본 비교에서는 먼저 `MODEL_ID`만 바꾸고 나머지 조건을 고정합니다. 여러 조건을 동시에 바꾸면 성능 차이가 모델 때문인지 학습 조건 때문인지 구분하기 어렵습니다.

In [ ]:
# ============================================================
# 초보자는 아래 두 줄만 변경하면 됩니다.
# ============================================================
MODEL_ID = "resnet18"
QUICK_RUN = True

# 운영체제에 맞는 데이터·체크포인트·결과 저장 위치를 준비합니다.
paths = get_lab_paths(create=True)
data_root = str(resolve_data_dir())

if QUICK_RUN:
    # 빠른 체험 모드: 전체 흐름을 짧은 시간 안에 확인합니다.
    data_config = DataConfig(
        batch_size=64,
        val_ratio=0.1,
        seed=42,
        data_root=data_root,
        max_train_samples=2048,
        max_val_samples=512,
        max_test_samples=512,
    )
    train_config = TrainConfig(epochs=1, learning_rate=0.001)
    weight_dir = str(paths.checkpoints_dir / "quick")
else:
    # 전체 실험 모드: Train 45,000장, Validation 5,000장, Test 10,000장을 사용합니다.
    data_config = DataConfig(
        batch_size=64, val_ratio=0.1, seed=42, data_root=data_root
    )
    train_config = TrainConfig(epochs=10, learning_rate=0.001)
    weight_dir = str(paths.checkpoints_dir / "full")

# 아래 객체 하나가 모델, 데이터, 학습, 장치와 저장 위치를 모두 관리합니다.
config = ExperimentConfig(
    model_id=MODEL_ID,
    num_classes=10,          # CIFAR-10은 클래스가 10개입니다.
    image_size=32,           # CIFAR-10 이미지 크기는 32x32입니다.
    device="auto",        # CUDA → Apple MPS → CPU 순으로 자동 선택합니다.
    weight_dir=weight_dir,
    data=data_config,
    train=train_config,
)

# 데이터 분할뿐 아니라 모델 초기화와 학습 난수도 동일하게 고정합니다.
set_global_seed(config.data.seed, deterministic=True)
RUN_ID = experiment_id(config)

mode_name = "빠른 체험" if QUICK_RUN else "전체 학습"
print(f"실행 모드: {mode_name}")
print(f"선택 모델: {config.model_id}")
print(f"학습 epoch: {config.train.epochs}")
print(f"데이터 위치: {config.data.data_root}")
print(f"체크포인트 위치: {config.weight_dir}")
print(f"실험 ID: {RUN_ID}")

### 설정 셀 결과 확인

출력된 `RUN_ID`는 현재 설정을 요약한 고유 식별자입니다. `epochs`만 늘려도 같은 실험을 이어서 학습할 수 있도록 epoch 목표는 ID 계산에서 제외됩니다. 반면 seed나 학습률을 바꾸면 새로운 실험 ID와 체크포인트가 만들어집니다.

### 선택 가능한 모델 확인하기

아래 표는 현재 `32×32` CIFAR-10 파이프라인에서 바로 사용할 수 있는 모델 ID를 보여 줍니다.

- `MODEL ID`: 설정에 그대로 입력할 문자열입니다.
- `FAMILY`: CNN, Transformer 등 모델 계열입니다.
- `INSTALL GROUP`: 추가 의존성 그룹입니다. `transformers` 모델은 전체 노트북 설치 구성이 필요합니다.

모델 이름을 직접 입력할 때는 표의 ID와 정확히 일치해야 합니다.

In [ ]:
# 표의 MODEL ID 값을 설정 셀의 MODEL_ID에 그대로 사용할 수 있습니다.
print(format_model_catalog(cifar10_ready_only=True))

### 모델 선택 기준

처음에는 대표 모델 몇 개만 비교하세요. 모든 모델을 한꺼번에 실행하면 학습 시간과 저장 공간이 크게 늘어납니다. 먼저 Quick run으로 실행 가능 여부를 확인한 뒤 목적에 맞는 후보만 전체 학습으로 확장하는 것이 좋습니다.

## 2. 모델 생성과 구조 확인

Registry가 `MODEL_ID`에 맞는 생성 함수를 찾아 모델을 만듭니다. `torchinfo.summary` 표는 다음 순서로 읽습니다.

1. 각 층의 출력 shape이 예상대로 이어지는지 확인합니다.
2. 합성곱 계열에서는 채널 수와 공간 크기의 변화를 확인합니다.
3. `Param #` 열에서 파라미터가 많은 구간을 찾습니다.
4. 마지막 출력이 `[1, 10]`인지 확인합니다. 이미지 1장에 대한 10개 클래스 점수라는 뜻입니다.

이 단계에서는 아직 학습하지 않았기 때문에 출력값 자체에는 의미가 없습니다. 구조와 텐서 연결이 올바른지를 검사하는 단계입니다.

In [ ]:
# 아직 학습 전이므로 정확도보다 shape과 파라미터 연결을 확인합니다.
# MODEL_ID에 해당하는 모델을 생성합니다.
model = create_model(
    config.model_id,
    num_classes=config.num_classes,
    image_size=config.image_size,
)

# 한 장의 가상 입력을 넣어 계층별 출력 크기와 파라미터 수를 확인합니다.
summary(
    model,
    input_size=(1, 3, config.image_size, config.image_size),
)

### 구조 출력에서 자주 발견하는 문제

- 마지막 출력 클래스 수가 10이 아님
- 공간 크기가 너무 빨리 0 또는 1로 줄어듦
- 입력 채널 수가 3과 맞지 않음
- 분류기 앞 특징 수가 예상과 다름

이 노트북의 Registry 모델들은 자동 검증을 거치지만, 모델을 직접 추가할 때는 이 요약표를 먼저 확인하세요.

## 3. 실행 장치 확인

모델과 입력 텐서는 같은 장치에 있어야 합니다. 이 프로젝트는 사용 가능한 장치를 다음 우선순위로 선택합니다.

1. NVIDIA GPU가 있으면 CUDA
2. Apple Silicon 환경이면 MPS
3. 그 외에는 CPU

Windows에서는 여러 worker가 노트북 프로세스를 다시 실행하는 문제를 피하기 위해 `num_workers=0`을 기본값으로 사용합니다. `pin_memory`는 CUDA로 데이터를 옮길 때 전송 효율을 높이는 옵션입니다.

장치가 CPU로 표시되어도 오류가 아닙니다. 같은 코드를 사용할 수 있지만 큰 모델과 전체 데이터 학습에는 시간이 더 걸립니다.

In [ ]:
# 이후 생성되는 입력 배치도 반드시 이 장치로 이동해야 합니다.
# 현재 컴퓨터에 맞는 장치와 DataLoader 옵션을 자동으로 결정합니다.
runtime = detect_environment(config.device, config.data.num_workers)
device = runtime.device

# 모델 파라미터를 선택한 장치로 이동합니다.
model = model.to(device)
print(f"실행 환경: {runtime}")

### 장치 확인 방법

출력의 `device=cpu`는 CPU 실행, `device=cuda`는 NVIDIA GPU 실행을 뜻합니다. 모델만 GPU에 있고 데이터가 CPU에 있으면 오류가 발생하므로 학습 반복문에서 각 배치를 같은 장치로 이동합니다.

## 4. 데이터 분할과 이미지 확인

CIFAR-10은 10개 클래스의 `32×32` 컬러 이미지 데이터셋입니다. 데이터는 용도에 따라 엄격히 나눕니다.

| 분할 | 용도 | 가중치 갱신에 사용? |
|---|---|---|
| Train | 모델 파라미터 학습 | 예 |
| Validation | epoch별 모델 선택과 과적합 확인 | 아니요 |
| Test | 모든 선택이 끝난 뒤 최종 평가 | 아니요 |

Train에는 무작위 crop과 좌우 반전 같은 augmentation을 적용할 수 있습니다. 이는 새로운 정답을 만드는 것이 아니라 같은 이미지의 허용 가능한 변형을 보여 주어 일반화를 돕습니다. Validation과 Test에는 무작위 augmentation을 적용하지 않아 평가 조건을 일정하게 유지합니다.

Test 결과를 보고 설정을 반복해서 바꾸면 Test가 사실상 Validation처럼 사용되는 데이터 누수가 생길 수 있습니다.

In [ ]:
# 설정값에 따라 CIFAR-10을 Train/Validation/Test로 준비합니다.
# 데이터가 없다면 첫 실행에서 자동으로 다운로드합니다.
trainloader, valloader, testloader, classes = load_cifar10_data(
    batch_size=config.data.batch_size,
    val_ratio=config.data.val_ratio,
    seed=config.data.seed,
    num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory,
    image_size=config.image_size,
    data_root=config.data.data_root,
    max_train_samples=config.data.max_train_samples,
    max_val_samples=config.data.max_val_samples,
    max_test_samples=config.data.max_test_samples,
)

# 분할 크기와 실제 학습 이미지를 그림으로 확인합니다.
visualize_data_overview(trainloader, valloader, testloader, classes)

### 데이터 그림 읽기

분할 크기의 합이 예상 데이터 수와 맞는지 먼저 확인합니다. Train 샘플이 약간 이동하거나 뒤집혀 보이는 것은 augmentation 때문입니다. 클래스 이름과 이미지가 명백히 맞지 않는다면 데이터 인덱스나 라벨 처리 과정을 점검해야 합니다.

## 5. 학습하고 Validation으로 최적 모델 선택

한 epoch의 내부 과정은 다음과 같습니다.

1. Train 배치를 모델에 입력해 클래스 점수를 계산합니다.
2. Cross-entropy loss로 예측과 정답의 차이를 측정합니다.
3. 역전파로 각 파라미터의 기울기를 구합니다.
4. Optimizer가 학습률만큼 파라미터를 갱신합니다.
5. epoch가 끝나면 가중치를 고정하고 Validation 성능을 측정합니다.
6. 지금까지 가장 높은 Validation 정확도의 가중치를 저장합니다.

실험 ID에는 모델, seed, 학습률, 데이터 설정이 반영됩니다. 다른 조건의 실험이 같은 파일을 덮어쓰지 않으며, 같은 설정의 체크포인트가 있으면 재사용합니다.

- `checkpoint is None`: 처음부터 학습합니다.
- 기존 체크포인트 존재: 저장된 최적 모델을 불러옵니다.
- CLI의 `--resume`: 마지막 epoch의 가중치와 optimizer 상태에서 이어서 학습합니다.
- CLI의 `--retrain`: 기존 결과와 관계없이 처음부터 다시 학습합니다.

In [ ]:
# 모델·seed·학습률·데이터 설정이 같은 실험 체크포인트만 불러옵니다.
checkpoint = load_model_weights(
    model,
    config.model_id,
    device=device,
    weight_dir=config.weight_dir,
    experiment_id=RUN_ID,
    expected_config=config,
)

if checkpoint is None:
    # 저장된 모델이 없을 때만 처음부터 학습합니다.
    history = train_model(
        model,
        trainloader,
        valloader,
        device,
        epochs=config.train.epochs,
        learning_rate=config.train.learning_rate,
        model_id=config.model_id,
        weight_dir=config.weight_dir,
        experiment_config=config,
        experiment_id=RUN_ID,
    )
else:
    # 저장된 학습 이력도 함께 복원합니다.
    history = checkpoint.get("history")

# Train/Validation의 loss와 accuracy 변화를 그래프로 확인합니다.
plot_training_history(history)

### 학습 곡선 진단표

| 관찰 | 가능한 해석 | 다음 확인 |
|---|---|---|
| Train·Validation loss 모두 감소 | 정상 학습 가능성 | 더 학습했을 때 계속 개선되는지 확인 |
| Train만 개선, Validation 악화 | 과적합 가능성 | augmentation, 정규화, 조기 종료 검토 |
| 둘 다 거의 변하지 않음 | 학습률·모델·데이터 문제 가능성 | loss와 기울기, 라벨 확인 |
| loss가 크게 진동하거나 NaN | 학습률 과대 또는 수치 불안정 | 학습률 축소, 입력 범위 확인 |

짧은 Quick run에서는 곡선이 매끄럽지 않을 수 있습니다. 여러 epoch와 전체 데이터에서 반복되는 경향인지 확인해야 합니다.

## 6. Test 데이터로 마지막 평가

학습과 모델 선택이 끝난 뒤 Test 데이터로 한 번 평가합니다. 전체 정확도만 보면 어떤 종류의 오류가 많은지 알 수 없으므로 세 결과를 함께 봅니다.

- **전체 정확도**: 모든 Test 이미지 중 맞힌 비율입니다.
- **클래스별 정확도**: 특정 클래스에 성능이 치우쳤는지 보여 줍니다.
- **Confusion matrix**: 실제 클래스와 예측 클래스의 조합별 개수를 보여 줍니다.

Confusion matrix의 행은 실제 클래스, 열은 예측 클래스입니다. 대각선은 정답이고 대각선 밖의 큰 값은 자주 혼동하는 클래스 쌍입니다. 예를 들어 `cat` 행의 `dog` 열이 크면 실제 고양이를 개로 잘못 분류한 경우가 많다는 뜻입니다.

In [ ]:
# Test 결과는 모델과 설정 선택이 모두 끝난 뒤 최종 보고에 사용합니다.
# 최적 Validation checkpoint 상태의 모델을 Test 데이터로 평가합니다.
test_result = evaluate_model_detailed(
    model,
    testloader,
    device,
    num_classes=config.num_classes,
)

print(f"최종 Test accuracy: {test_result['accuracy']:.2f}%")
plot_test_results(test_result, classes)

### 최종 결과를 해석할 때

정확도 차이가 작다면 한 번의 seed 결과만으로 우열을 단정하지 않습니다. 여러 seed로 반복한 평균과 변동성을 확인하는 것이 더 신뢰할 수 있습니다. 또한 교육용 Quick run은 적은 표본만 사용하므로 전체 데이터 결과와 순위가 달라질 수 있습니다.

## 다음 실험 아이디어와 공정한 비교 방법

전체 과정이 정상 실행되었다면 한 번에 하나의 요인만 바꾸어 결과를 기록하세요.

1. `perceptron → mlp → alexnet → resnet18` 순서로 표현력의 발전을 비교합니다.
2. `mobilenet_v2`로 바꾸어 작은 모델의 정확도와 속도를 확인합니다.
3. `vit_tiny`로 바꾸어 합성곱과 Transformer의 입력 처리 차이를 비교합니다.
4. Quick run이 끝나면 `QUICK_RUN=False`로 전체 데이터 학습을 실행합니다.

### 실험 기록 체크리스트

- seed, 데이터 분할, epoch, 학습률이 동일한가?
- 최고 Validation 성능의 체크포인트를 비교했는가?
- Test를 모델 선택에 반복 사용하지 않았는가?
- 정확도뿐 아니라 파라미터 수, 추론 시간, 체크포인트 크기도 기록했는가?
- 한 번의 실행 결과를 일반적인 결론으로 과대 해석하지 않았는가?

명령행에서는 다음 명령으로 여러 모델의 CSV·JSON·PNG 비교 보고서를 한 번에 만들 수 있습니다.

```powershell
cifar10-lab compare --models perceptron mlp alexnet resnet18 --quick
```